In [49]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [50]:
IMG_SIZE = 128
BATCH_SIZE = 16
NUM_CLASSES = 7

In [51]:
#Load Dataset

train_df = pd.read_csv("Datasets/train.csv")
valid_df = pd.read_csv("Datasets/val.csv")
test_df = pd.read_csv("Datasets/test.csv")

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)
metadata = pd.read_csv("Datasets/HAM10000_metadata.csv")

metadata.head()

(7010, 8)
(1502, 8)
(1503, 8)


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [52]:
image_dir1 = "Datasets/HAM10000_images_part_1"
image_dir2 = "Datasets/HAM10000_images_part_2"

image_path = {}

for folder in [image_dir1, image_dir2]:

    for file in os.listdir(folder):

        image_id = file.split(".")[0]

        image_path[image_id] = os.path.join(folder, file)

metadata["path"] = metadata["image_id"].map(image_path)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2\ISIC_0031633.jpg


In [53]:
print("Missing Paths :", metadata["path"].isna().sum())

metadata[["image_id", "path"]].head()

Missing Paths : 0


,image_id,path
0,ISIC_0027419,Datasets/HAM10000_images_part_1\ISIC_0027419.jpg
1,ISIC_0025030,Datasets/HAM10000_images_part_1\ISIC_0025030.jpg
2,ISIC_0026769,Datasets/HAM10000_images_part_1\ISIC_0026769.jpg
3,ISIC_0025661,Datasets/HAM10000_images_part_1\ISIC_0025661.jpg
4,ISIC_0031633,Datasets/HAM10000_images_part_2\ISIC_0031633.jpg


In [54]:
encoder = LabelEncoder()

metadata["label"] = encoder.fit_transform(metadata["dx"])

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0027419.jpg,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0025030.jpg,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0026769.jpg,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1\ISIC_0025661.jpg,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2\ISIC_0031633.jpg,2


In [55]:
train_df, temp_df = train_test_split(

    metadata,

    test_size=0.30,

    stratify=metadata["label"],

    random_state=42
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7010, 9)
(1502, 9)
(1503, 9)


In [56]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

In [57]:
train_df["path"] = train_df["path"].astype(str)
val_df["path"] = val_df["path"].astype(str)
test_df["path"] = test_df["path"].astype(str)

In [58]:
classes = np.unique(train_df["label"])

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=train_df["label"]
)

class_weights = dict(zip(range(len(classes)), weights))

print(class_weights)

{0: 4.37305053025577, 1: 2.7817460317460316, 2: 1.3022478172023035, 3: 12.36331569664903, 4: 1.285530900421786, 5: 0.21338772031292808, 6: 10.115440115440116}


In [59]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,

    height_shift_range=0.1,

    zoom_range=0.2,

    horizontal_flip=True,

    vertical_flip=True
)

test_datagen = ImageDataGenerator(

    rescale=1./255
)

In [60]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [61]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    horizontal_flip=True,

    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]

)

test_datagen = ImageDataGenerator(

    rescale=1./255

)

In [62]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True

)

Found 7010 validated image filenames belonging to 7 classes.


In [63]:
#Build MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense
 
base_model = MobileNetV2(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
 
)
 
base_model.trainable = False
 
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
 
x = base_model(inputs, training=False)
 
x = GlobalAveragePooling2D()(x)
 
x = Dense(256, activation="relu")(x)
 
outputs = Dense(NUM_CLASSES, activation="softmax")(x)
 
mobilenet_model = Model(inputs, outputs)
 
mobilenet_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)           │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_128 (Functional)    │ (None, 4, 4, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [66]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True,

    verbose=1

)

In [67]:
from tensorflow.keras.optimizers import Adam

mobilenet_model.compile(

    optimizer=Adam(learning_rate=0.001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [68]:
history_es = mobilenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=15,

    callbacks=[early_stop]

)

Epoch 1/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 241s 527ms/step - accuracy: 0.6861 - loss: 0.9798 - val_accuracy: 0.7230 - val_loss: 0.7584
Epoch 2/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 232s 459ms/step - accuracy: 0.7337 - loss: 0.7405 - val_accuracy: 0.7210 - val_loss: 0.7976
Epoch 3/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 205s 467ms/step - accuracy: 0.7580 - loss: 0.6791 - val_accuracy: 0.7244 - val_loss: 0.7960
Epoch 4/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 229s 522ms/step - accuracy: 0.7403 - loss: 0.6926 - val_accuracy: 0.7463 - val_loss: 0.7108
Epoch 5/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 208s 473ms/step - accuracy: 0.7551 - loss: 0.6773 - val_accuracy: 0.7423 - val_loss: 0.7424
Epoch 6/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 225s 512ms/step - accuracy: 0.7723 - loss: 0.6317 - val_accuracy: 0.7071 - val_loss: 0.8362
Epoch 7/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 210s 479ms/step - accuracy: 0.7763 - loss: 0.6347 - val_accuracy: 0.7410 - val_loss: 0.7823
Epoch 7: early stopping
Restoring model weights from the end of the best epo

In [69]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = mobilenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc =mobilenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc =mobilenet_model.evaluate(test_generator, verbose=0)

pred = mobilenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [70]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.7646219730377197, 0.7463381886482239, 0.7332002520561218, 0.723558756555672, 0.7332002661343978, 0.7163950707497758]


In [71]:
best_model.save("moilenet_ES.keras")

# LR

In [72]:
#Build MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense
 
base_model = MobileNetV2(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
 
)
 
base_model.trainable = False
 
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
 
x = base_model(inputs, training=False)
 
x = GlobalAveragePooling2D()(x)
 
x = Dense(256, activation="relu")(x)
 
outputs = Dense(NUM_CLASSES, activation="softmax")(x)
 
mobilenet_model = Model(inputs, outputs)
 
mobilenet_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)           │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_128 (Functional)    │ (None, 4, 4, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_3           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [77]:
from tensorflow.keras.optimizers import Adam

mobilenet_model.compile(

    optimizer=Adam(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [78]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-7,

    verbose=1

)

In [79]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=2,

    restore_best_weights=True,

    verbose=1

)

In [80]:
history_lr = mobilenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=10,

    callbacks=[lr_scheduler, early_stop]

)

Epoch 1/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 233s 509ms/step - accuracy: 0.6756 - loss: 0.9635 - val_accuracy: 0.5945 - val_loss: 1.0398 - learning_rate: 5.0000e-04
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 212s 483ms/step - accuracy: 0.7392 - loss: 0.7291 - val_accuracy: 0.7350 - val_loss: 0.7311 - learning_rate: 5.0000e-04
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 267s 494ms/step - accuracy: 0.7514 - loss: 0.6942 - val_accuracy: 0.7204 - val_loss: 0.7905 - learning_rate: 5.0000e-04
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step - accuracy: 0.7502 - loss: 0.6774
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
439/439 ━━━━━━━━━━━━━━━━━━━━ 225s 513ms/step - accuracy: 0.7502 - loss: 0.6774 - val_accuracy: 0.7310 - val_loss: 0.7741 - learning_rate: 5.0000e-04
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 2.


In [81]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = mobilenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc =mobilenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc =mobilenet_model.evaluate(test_generator, verbose=0)

pred = mobilenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [82]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.7509272694587708, 0.7350199818611145, 0.7132402062416077, 0.73942105872712, 0.7132401862940785, 0.6934790029387732]


In [83]:
best_model.save("moilenet_LR.keras")

# SGD

In [84]:
#Build MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense
 
base_model = MobileNetV2(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
 
)
 
base_model.trainable = False
 
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
 
x = base_model(inputs, training=False)
 
x = GlobalAveragePooling2D()(x)
 
x = Dense(256, activation="relu")(x)
 
outputs = Dense(NUM_CLASSES, activation="softmax")(x)
 
mobilenet_model = Model(inputs, outputs)
 
mobilenet_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)           │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_128 (Functional)    │ (None, 4, 4, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_4           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [86]:
from tensorflow.keras.optimizers import Adam

mobilenet_model.compile(

    optimizer=SGD(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [87]:
history_sg = mobilenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=8,

    callbacks=[lr_scheduler, early_stop]

)

Epoch 1/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 242s 530ms/step - accuracy: 0.5784 - loss: 1.3743 - val_accuracy: 0.6731 - val_loss: 0.9743 - learning_rate: 5.0000e-04
Epoch 2/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 159s 361ms/step - accuracy: 0.6627 - loss: 0.9781 - val_accuracy: 0.6884 - val_loss: 0.9087 - learning_rate: 5.0000e-04
Epoch 3/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 146s 333ms/step - accuracy: 0.6944 - loss: 0.8734 - val_accuracy: 0.7011 - val_loss: 0.8735 - learning_rate: 5.0000e-04
Epoch 4/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 149s 340ms/step - accuracy: 0.7086 - loss: 0.8632 - val_accuracy: 0.7037 - val_loss: 0.8544 - learning_rate: 5.0000e-04
Epoch 5/8
439/439 ━━━━━━━━━━━━━━━━━━━━ 150s 342ms/step - accuracy: 0.7012 - loss: 0.8532 - val_accuracy: 0.7051 - val_loss: 0.8405 - learning_rate: 5.0000e-04
Epoch 6/8
288/439 ━━━━━━━━━━━━━━━━━━━━ 45s 301ms/step - accuracy: 0.7066 - loss: 0.8331

KeyboardInterrupt: 

In [88]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = mobilenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc =mobilenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc =mobilenet_model.evaluate(test_generator, verbose=0)

pred = mobilenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [89]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.7132667899131775, 0.712383508682251, 0.6946107745170593, 0.6310128905323807, 0.6946107784431138, 0.6504102751589164]


In [90]:
best_model.save("moilenet_sg.keras")

# RMS

In [91]:
#Build MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense
 
base_model = MobileNetV2(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
 
)
 
base_model.trainable = False
 
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
 
x = base_model(inputs, training=False)
 
x = GlobalAveragePooling2D()(x)
 
x = Dense(256, activation="relu")(x)
 
outputs = Dense(NUM_CLASSES, activation="softmax")(x)
 
mobilenet_model = Model(inputs, outputs)
 
mobilenet_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)           │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_128 (Functional)    │ (None, 4, 4, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_5           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [92]:
from tensorflow.keras.optimizers import RMSprop

mobilenet_model.compile(

    optimizer=RMSprop(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [93]:
history_rms = mobilenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    callbacks=[lr_scheduler, early_stop]

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 163s 356ms/step - accuracy: 0.6759 - loss: 0.9868 - val_accuracy: 0.7277 - val_loss: 0.8423 - learning_rate: 5.0000e-04
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 150s 342ms/step - accuracy: 0.7287 - loss: 0.7571 - val_accuracy: 0.6944 - val_loss: 0.8544 - learning_rate: 5.0000e-04
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 144s 327ms/step - accuracy: 0.7366 - loss: 0.7375 - val_accuracy: 0.7290 - val_loss: 0.7546 - learning_rate: 5.0000e-04
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 143s 326ms/step - accuracy: 0.7527 - loss: 0.6982 - val_accuracy: 0.7510 - val_loss: 0.7214 - learning_rate: 5.0000e-04
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 919s 2s/step - accuracy: 0.7664 - loss: 0.6538 - val_accuracy: 0.7237 - val_loss: 0.8116 - learning_rate: 5.0000e-04
Restoring model weights from the end of the best epoch: 4.


In [94]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = mobilenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc =mobilenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc =mobilenet_model.evaluate(test_generator, verbose=0)

pred = mobilenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [95]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.7661911845207214, 0.7509986758232117, 0.7398536205291748, 0.7160304690389595, 0.739853626081171, 0.7164821673227141]


In [96]:
best_model.save("moilenet_rms.keras")

# Batch Comparison

In [97]:
train_generator_8 = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=True

)

val_generator_8 = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

test_generator_8 = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [98]:
#Build MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense
 
base_model = MobileNetV2(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
 
)
 
base_model.trainable = False
 
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
 
x = base_model(inputs, training=False)
 
x = GlobalAveragePooling2D()(x)
 
x = Dense(256, activation="relu")(x)
 
outputs = Dense(NUM_CLASSES, activation="softmax")(x)
 
mobilenet_model = Model(inputs, outputs)
 
mobilenet_model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)          │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_128 (Functional)    │ (None, 4, 4, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_6           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [99]:
from tensorflow.keras.optimizers import Adam

mobilenet_model.compile(

    optimizer=Adam(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [101]:
history_rms = mobilenet_model.fit(

    train_generator_8,

    validation_data=val_generator_8,

    epochs=5,

    callbacks=[lr_scheduler, early_stop]

)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - accuracy: 0.6736 - loss: 0.9877 - val_accuracy: 0.7104 - val_loss: 0.7678 - learning_rate: 5.0000e-04
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 250s 951ms/step - accuracy: 0.7361 - loss: 0.7396 - val_accuracy: 0.7417 - val_loss: 0.7176 - learning_rate: 5.0000e-04
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 206s 938ms/step - accuracy: 0.7548 - loss: 0.6772 - val_accuracy: 0.7390 - val_loss: 0.7452 - learning_rate: 5.0000e-04
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 838ms/step - accuracy: 0.7495 - loss: 0.6824
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
220/220 ━━━━━━━━━━━━━━━━━━━━ 212s 964ms/step - accuracy: 0.7496 - loss: 0.6823 - val_accuracy: 0.7170 - val_loss: 0.7571 - learning_rate: 5.0000e-04
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 2.


In [102]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = mobilenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc =mobilenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc =mobilenet_model.evaluate(test_generator, verbose=0)

pred = mobilenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [103]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.762767493724823, 0.7416777610778809, 0.7405189871788025, 0.7225366812193159, 0.7405189620758483, 0.7087629385186853]


In [104]:
best_model.save("moilenet_batch_size.keras")

# HyperParameter

In [105]:
import keras_tuner as kt

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop

In [106]:
def build_mobilenet(hp):

    base_model = MobileNetV2(

        weights="imagenet",

        include_top=False,

        input_shape=(IMG_SIZE, IMG_SIZE, 3)

    )

    base_model.trainable = False

    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = base_model(inputs, training=False)

    x = GlobalAveragePooling2D()(x)

    x = Dense(

        units=hp.Choice(
            "dense_units",
            [128,256]
        ),

        activation="relu"

    )(x)

    x = Dropout(

        hp.Choice(
            "dropout",
            [0.3,0.5]
        )

    )(x)

    outputs = Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    model = Model(inputs, outputs)

    optimizer_name = hp.Choice(
        "optimizer",
        ["adam","rmsprop"]
    )

    lr = hp.Choice(
        "learning_rate",
        [0.001,0.0001]
    )

    if optimizer_name == "adam":
        optimizer = Adam(learning_rate=lr)
    else:
        optimizer = RMSprop(learning_rate=lr)

    model.compile(

        optimizer=optimizer,

        loss="categorical_crossentropy",

        metrics=["accuracy"]

    )

    return model

In [107]:
tuner = kt.RandomSearch(

    build_mobilenet,

    objective="val_accuracy",

    max_trials=3,

    directory="mobilenet_tuning",

    project_name="mobilenet_hp"

)

In [108]:
tuner.search(

    train_generator,

    validation_data=val_generator,

    epochs=5

)

Trial 3 Complete [00h 13m 50s]
val_accuracy: 0.7496671080589294

Best val_accuracy So Far: 0.7503328919410706
Total elapsed time: 00h 44m 21s


In [109]:
best_hps = tuner.get_best_hyperparameters(1)[0]

print(best_hps.values)

{'dense_units': 512, 'dropout': 0.2, 'optimizer': 'adam', 'learning_rate': 0.0001}


In [110]:
best_model = tuner.hypermodel.build(
    best_hps
)

history = best_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5
)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 182s 399ms/step - accuracy: 0.6512 - loss: 1.0647 - val_accuracy: 0.7217 - val_loss: 0.7630
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 155s 352ms/step - accuracy: 0.7270 - loss: 0.7718 - val_accuracy: 0.7164 - val_loss: 0.7613
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 163s 371ms/step - accuracy: 0.7330 - loss: 0.7466 - val_accuracy: 0.7270 - val_loss: 0.7583
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 161s 367ms/step - accuracy: 0.7433 - loss: 0.7074 - val_accuracy: 0.7284 - val_loss: 0.7470
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 163s 372ms/step - accuracy: 0.7532 - loss: 0.6743 - val_accuracy: 0.7510 - val_loss: 0.7127


In [111]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = best_model.evaluate(train_generator, verbose=0)

val_loss, val_acc =best_model.evaluate(val_generator, verbose=0)

test_loss, test_acc =best_model.evaluate(test_generator, verbose=0)

pred = best_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [112]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.7756062746047974, 0.7509986758232117, 0.7305389046669006, 0.7386005773968867, 0.7305389221556886, 0.713583029766733]


In [113]:
best_model.save("best_model_mobilenet.keras")

# FineTuning

In [114]:
base_model.trainable = False

In [115]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

In [116]:
base_model.trainable = True

for layer in base_model.layers[:-10]:

    layer.trainable = False

In [117]:
print("Trainable Layers:")

for layer in base_model.layers[-10:]:

    print(layer.name, layer.trainable)

Trainable Layers:
block_16_expand_BN True
block_16_expand_relu True
block_16_depthwise True
block_16_depthwise_BN True
block_16_depthwise_relu True
block_16_project True
block_16_project_BN True
Conv_1 True
Conv_1_bn True
out_relu True


In [118]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D

inputs = Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = base_model(
    inputs,
    training=False
)

x = GlobalAveragePooling2D()(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.3)(x)

outputs = Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

mobilenet_ft = Model(
    inputs,
    outputs
)

mobilenet_ft.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)           │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_128 (Functional)    │ (None, 4, 4, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 1,062,215 (4.05 MB)

 Non-trainable params: 1,525,504 (5.82 MB)

In [119]:
from tensorflow.keras.optimizers import Adam

mobilenet_ft.compile(

    optimizer=Adam(
        learning_rate=1e-5
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [120]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=2,

    restore_best_weights=True,

    verbose=1

)

In [121]:
history_ft = mobilenet_ft.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    callbacks=[early_stop]

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 183s 390ms/step - accuracy: 0.2203 - loss: 2.1265 - val_accuracy: 0.2696 - val_loss: 1.8313
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 163s 372ms/step - accuracy: 0.3132 - loss: 1.7374 - val_accuracy: 0.4854 - val_loss: 1.4288
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 167s 381ms/step - accuracy: 0.3931 - loss: 1.5473 - val_accuracy: 0.5160 - val_loss: 1.3161
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 163s 371ms/step - accuracy: 0.4172 - loss: 1.3992 - val_accuracy: 0.5300 - val_loss: 1.2499
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 165s 376ms/step - accuracy: 0.4568 - loss: 1.3394 - val_accuracy: 0.4794 - val_loss: 1.3228
Restoring model weights from the end of the best epoch: 4.


In [122]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = mobilenet_ft.evaluate(train_generator, verbose=0)

val_loss, val_acc =mobilenet_ft.evaluate(val_generator, verbose=0)

test_loss, test_acc =mobilenet_ft.evaluate(test_generator, verbose=0)

pred = mobilenet_ft.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [123]:
best_model.save("fine_mobilenet.keras")

In [124]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.5573466420173645, 0.529960036277771, 0.5262807607650757, 0.6952514463527042, 0.5262807717897539, 0.5842289437487483]


In [128]:
import pandas as pd

mobilenet_phase5 = pd.DataFrame({

    "Technique": [
        "Early Stopping",
        "Learning Rate Scheduling",
        "SGD",
        "RMSprop",
        "Batch Size = 32",
        "Hyperparameter Tuning",
        "Fine Tuning"
    ],

    "Train Accuracy": [
        0.7646219730377197,
        0.7509272694587708,
        0.7132667899131775,
        0.7661911845207214,
        0.762767493724823,
        0.7756062746047974,
        0.5573466420173645
    ],

    "Validation Accuracy": [
        0.7463381886482239,
        0.7350199818611145,
        0.712383508682251,
        0.7509986758232117,
        0.7416777610778809,
        0.7509986758232117,
        0.529960036277771
    ],

    "Test Accuracy": [
        0.7332002520561218,
        0.7132402062416077,
        0.6946107745170593,
        0.7398536205291748,
        0.7405189871788025,
        0.7305389046669006,
        0.5262807607650757
    ],

    "Precision": [
        0.723558756555672,
        0.73942105872712,
        0.6310128905323807,
        0.7160304690389595,
        0.7225366812193159,
        0.7386005773968867,
        0.6952514463527042
    ],

    "Recall": [
        0.7332002661343978,
        0.7132401862940785,
        0.6946107784431138,
        0.739853626081171,
        0.7405189620758483,
        0.7305389221556886,
        0.5262807717897539
    ],

    "F1 Score": [
        0.7163950707497758,
        0.6934790029387732,
        0.6504102751589164,
        0.7164821673227141,
        0.7087629385186853,
        0.713583029766733,
        0.5842289437487483
    ]
})

mobilenet_phase5 = mobilenet_phase5.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

mobilenet_phase5

,Technique,Train Accuracy,Validation Accuracy,Test Accuracy,Precision,Recall,F1 Score
0,Batch Size = 32,0.762767,0.741678,0.740519,0.722537,0.740519,0.708763
1,RMSprop,0.766191,0.750999,0.739854,0.716030,0.739854,0.716482
2,Early Stopping,0.764622,0.746338,0.733200,0.723559,0.733200,0.716395
3,Hyperparameter Tuning,0.775606,0.750999,0.730539,0.738601,0.730539,0.713583
4,Learning Rate Scheduling,0.750927,0.735020,0.713240,0.739421,0.713240,0.693479
5,SGD,0.713267,0.712384,0.694611,0.631013,0.694611,0.650410
6,Fine Tuning,0.557347,0.529960,0.526281,0.695251,0.526281,0.584229
